# BTCUSDT Price Prediction - Complete Example

## Setup

Install once from the repo root (use `.venv` as the notebook kernel):

```bash
pip install -e ".[prediction]"
```

Imports use the `src` package (e.g. `from src.data.data_load import load_data`).

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import xgboost
import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Step 1: Load Data

In [18]:
from src.data.data_load import load_data

# Load BTCUSDT 1-minute kline data
df = load_data(
    symbols=['BTCUSDT'],
    start_date="2025-01-01",
    end_date="2025-04-30",
    interval="1m"
)

print(f"Data shape: {df.shape}")
print(f"\nData info:")
print(df.info())
print(f"\nFirst few rows:")
print(df.head())

Data shape: (171361, 13)

Data info:
<class 'pandas.DataFrame'>
RangeIndex: 171361 entries, 0 to 171360
Data columns (total 13 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   open_time        171361 non-null  datetime64[ms, UTC]
 1   open             171361 non-null  float64            
 2   high             171361 non-null  float64            
 3   low              171361 non-null  float64            
 4   close            171361 non-null  float64            
 5   volume           171361 non-null  float64            
 6   quote_volume     171361 non-null  float64            
 7   num_trades       171361 non-null  int64              
 8   taker_base_vol   171361 non-null  float64            
 9   taker_quote_vol  171361 non-null  float64            
 10  close_time       171361 non-null  int64              
 11  symbol           171361 non-null  str                
 12  year             171361 non-null

In [19]:
# Data statistics
print("Data statistics:")
print(df[['open', 'high', 'low', 'close', 'volume']].describe())

Data statistics:
                open           high            low          close  \
count  171361.000000  171361.000000  171361.000000  171361.000000   
mean    91656.563106   91692.126962   91620.840342   91656.567286   
std      7658.375883    7658.057689    7658.427395    7658.376088   
min     74610.000000   74737.820000   74508.000000   74610.000000   
25%     84399.510000   84421.990000   84377.350000   84399.520000   
50%     93437.500000   93470.180000   93404.970000   93437.490000   
75%     97346.430000   97375.260000   97313.770000   97346.430000   
max    109185.870000  109588.000000  108945.080000  109194.170000   

              volume  
count  171361.000000  
mean       19.237918  
std        34.073688  
min         0.112580  
25%         4.641750  
50%         9.684720  
75%        20.735520  
max      1899.034880  


## Step 2: Feature Engineering

In [20]:
from src.strategy.predictive import PriceFeatures, prepare_data_for_modeling

# Prepare features and target variable
print("Generating features...")
df_features, feature_cols, target_col = prepare_data_for_modeling(
    df,
    horizon=1,  # Predict 1 minute ahead
    lookback_window=60,
    dropna=True
)

print(f"\nDataset shape after feature engineering: {df_features.shape}")
print(f"Number of features: {len(feature_cols)}")
print(f"\nFeature columns: {feature_cols[:10]}...")  # Show first 10 features
print(f"\nTarget distribution:")
print(df_features[target_col].value_counts())
print(f"Target label balance: {df_features[target_col].mean():.2%} positive class")

Generating features...

Dataset shape after feature engineering: (171261, 76)
Number of features: 64

Feature columns: ['quote_volume', 'num_trades', 'taker_base_vol', 'taker_quote_vol', 'year', 'return', 'log_return', 'hl_ratio', 'co_ratio', 'cc_ratio']...

Target distribution:
target_binary
0    87038
1    84223
Name: count, dtype: int64
Target label balance: 49.18% positive class


In [21]:
# Display all feature columns
print(f"All features ({len(feature_cols)} total):")
for i, col in enumerate(feature_cols, 1):
    print(f"{i:3d}. {col}")

All features (64 total):
  1. quote_volume
  2. num_trades
  3. taker_base_vol
  4. taker_quote_vol
  5. year
  6. return
  7. log_return
  8. hl_ratio
  9. co_ratio
 10. cc_ratio
 11. ho_ratio
 12. lo_ratio
 13. sma_5
 14. sma_10
 15. sma_20
 16. sma_50
 17. sma_100
 18. ema_5
 19. ema_10
 20. ema_20
 21. ema_50
 22. ema_100
 23. momentum_5
 24. momentum_10
 25. momentum_20
 26. momentum_60
 27. rsi_14
 28. rsi_7
 29. macd
 30. macd_signal
 31. macd_hist
 32. bb_upper
 33. bb_middle
 34. bb_lower
 35. bb_width
 36. bb_position
 37. tr
 38. atr
 39. volume_sma_5
 40. volume_sma_10
 41. volume_sma_20
 42. volume_ratio
 43. obv
 44. obv_sma
 45. volatility_5
 46. volatility_10
 47. volatility_20
 48. volatility_60
 49. price_volume_corr
 50. close_lag_1
 51. return_lag_1
 52. volume_lag_1
 53. close_lag_2
 54. return_lag_2
 55. volume_lag_2
 56. close_lag_3
 57. return_lag_3
 58. volume_lag_3
 59. close_lag_4
 60. return_lag_4
 61. volume_lag_4
 62. close_lag_5
 63. return_lag_5
 64. vol

## Step 3: Data Preparation

In [22]:
from src.strategy.predictive import TimeSeriesSplitter

# Option 1: Split by date ranges
train_start = "2025-01-01"
train_end = "2025-03-31"
test_start = "2025-04-01"
test_end = "2025-04-30"

X_train, X_val, X_test, y_train, y_val, y_test = TimeSeriesSplitter.train_test_split_by_date(
    df_features,
    feature_cols,
    target_col,
    train_start_date=train_start,
    train_end_date=train_end,
    test_start_date=test_start,
    test_end_date=test_end,
    val_ratio=0.1  # 10% of training data for validation
)

# Option 2: Split by periods (days)
# X_train, X_val, X_test, y_train, y_val, y_test = TimeSeriesSplitter.train_test_split_by_period(
#     df_features,
#     feature_cols,
#     target_col,
#     train_period_days=365,  # 1 year training
#     test_period_days=90,    # 3 months testing
#     val_ratio=0.1
# )

Date-based time series split:
  Train: 115256 samples (2025-01-01 to 2025-03-31, 10.0% for validation)
  Val:   12806 samples (from training data)
  Test:  41760 samples (2025-04-01 to 2025-04-30)
  Total: 169822 samples



## Step 4: Train Models

In [23]:

import xgboost

In [24]:
from src.strategy.predictive import (
    LinearModel,
    RandomForestModel,
    GradientBoostingModel,
    XGBoostModel,
    MLPModel,
    ModelTrainer
)

# Initialize trainer
trainer = ModelTrainer()

# Initialize models
models = [
    LinearModel(),
    RandomForestModel(n_estimators=100, max_depth=15),
    XGBoostModel(n_estimators=100, max_depth=6, learning_rate=0.1),
]

# Train traditional ML models
for model in models:
    trainer.train_model(
        model,
        X_train, y_train,
        X_val, y_val
    )

Training Linear model...
Linear Model trained
Linear model trained successfully!

Training RandomForest model...
RandomForest Model trained
RandomForest model trained successfully!

Training XGBoost model...
XGBoost Model trained
XGBoost model trained successfully!



In [25]:
# Train MLP model (simple neural network)
mlp_model = MLPModel(hidden_layers=[128, 64, 32], epochs=20, batch_size=32)
trainer.train_model(
    mlp_model,
    X_train, y_train,
    X_val, y_val,
    verbose=1
)

Training MLP model...
Epoch 10/20, Loss: 0.6872
Epoch 20/20, Loss: 0.6863
MLP Model trained
MLP model trained successfully!



In [26]:
try:
    from src.strategy.predictive import CNNModel

    print("Training CNN model...")
    cnn_model = CNNModel(epochs=30, batch_size=32)

    cnn_model.train(
        X_train, y_train,
        seq_length=20,  # Use 20-minute lookback
        X_val=X_val, y_val=y_val,
        verbose=1
    )

    # Evaluate CNN
    trainer.models['CNN'] = cnn_model
    trainer.evaluate_model(cnn_model, X_test, y_test)

except ImportError as e:
    print(f"PyTorch not installed: {e}")
    print("Install with: pip install torch")

Training CNN model...
Epoch 10/30, Loss: 0.7459
Epoch 20/30, Loss: 0.4955
Epoch 30/30, Loss: 0.2782
CNN Model trained
Evaluating CNN model...

CNN - Evaluation Results
Accuracy:  0.5061
Precision: 0.4969
Recall:    0.4055
F1-Score:  0.4466

Confusion Matrix:
[[12810  8426]
 [12201  8323]]



## Step 5: Evaluate Models on Test Set

In [27]:
# Evaluate all trained models
for model_name, model in trainer.models.items():
    trainer.evaluate_model(model, X_test, y_test)

Evaluating Linear model...

Linear - Evaluation Results
Accuracy:  0.5118
Precision: 0.5046
Recall:    0.3621
F1-Score:  0.4216
AUC-ROC:   0.5121

Confusion Matrix:
[[13940  7296]
 [13093  7431]]

Evaluating RandomForest model...

RandomForest - Evaluation Results
Accuracy:  0.5121
Precision: 0.5035
Recall:    0.5130
F1-Score:  0.5082
AUC-ROC:   0.5190

Confusion Matrix:
[[10856 10380]
 [ 9996 10528]]

Evaluating XGBoost model...

XGBoost - Evaluation Results
Accuracy:  0.5129
Precision: 0.5044
Recall:    0.5089
F1-Score:  0.5066
AUC-ROC:   0.5186

Confusion Matrix:
[[10976 10260]
 [10080 10444]]

Evaluating MLP model...

MLP - Evaluation Results
Accuracy:  0.5137
Precision: 0.5233
Recall:    0.1184
F1-Score:  0.1931
AUC-ROC:   0.5185

Confusion Matrix:
[[19022  2214]
 [18094  2430]]

Evaluating CNN model...

CNN - Evaluation Results
Accuracy:  0.5061
Precision: 0.4969
Recall:    0.4055
F1-Score:  0.4466

Confusion Matrix:
[[12810  8426]
 [12201  8323]]



## Step 6: Model Comparison

In [32]:
# Compare all models
df_comparison = trainer.compare_models()


MODEL COMPARISON
       Model  Accuracy  Precision   Recall  F1-Score      AUC
RandomForest  0.512069   0.503539 0.512960  0.508206 0.519046
     XGBoost  0.512931   0.504444 0.508868  0.506646 0.518608
         CNN  0.506058   0.496925 0.405525  0.446597      NaN
      Linear  0.511758   0.504583 0.362064  0.421605 0.512131
        LSTM  0.512692   0.505952 0.360359  0.420921      NaN
         MLP  0.513697   0.523256 0.118398  0.193102 0.518481
 Transformer  0.509195   0.506076 0.056812  0.102155      NaN



## Step 7: Feature Importance (for tree-based models)

In [29]:
# Get feature importance from tree-based models
if 'RandomForest' in trainer.models:
    rf_model = trainer.models['RandomForest']
    rf_model.feature_cols = feature_cols
    
    importance_df = rf_model.feature_importance()
    print(f"\nRandom Forest - Top 20 Important Features:")
    print(importance_df.head(20))


Random Forest - Top 20 Important Features:
              feature  importance
62       return_lag_5    0.021380
26             rsi_14    0.021350
50       return_lag_1    0.020991
53       return_lag_2    0.020986
60       volume_lag_4    0.020704
56       return_lag_3    0.020584
24        momentum_20    0.020522
48  price_volume_corr    0.020411
63       volume_lag_5    0.020384
59       return_lag_4    0.020372
34           bb_width    0.020213
28               macd    0.020073
25        momentum_60    0.020010
57       volume_lag_3    0.019868
41       volume_ratio    0.019866
2      taker_base_vol    0.019860
35        bb_position    0.019849
29        macd_signal    0.019822
54       volume_lag_2    0.019816
23        momentum_10    0.019750


## Summary

### Popular Models for Price Prediction:

1. **Linear Models**
   - Logistic Regression: Simple, fast, interpretable baseline
   - Best for: Quick prototyping, understanding feature importance

2. **Tree-based Models**
   - Random Forest: Robust, handles non-linearity, resistant to overfitting
   - XGBoost: High accuracy, efficient, handles missing values well
   - Gradient Boosting: Strong ensemble method, good for structured data
   - Best for: High-dimensional feature data, good generalization

3. **Deep Learning**
   - MLP: Simple neural network, good for tabular data
   - LSTM: Captures long-term dependencies in sequences
   - CNN: Good for local patterns in sequential data
   - Transformer: State-of-the-art for sequences, captures complex patterns
   - Best for: Raw sequential data, learning temporal patterns

### Key Features for Price Prediction:

1. **Technical Indicators**: RSI, MACD, Bollinger Bands, ATR
2. **Moving Averages**: SMA, EMA of different periods
3. **Momentum**: Price momentum over different lookback periods
4. **Volume**: Volume-weighted indicators, OBV
5. **Volatility**: Standard deviation of returns
6. **Lagged Features**: Previous candle prices and returns
7. **Price Ratios**: HL ratio, CO ratio, etc.

### Recommendations:

- Start with **tree-based models** (Random Forest/XGBoost) for good balance of accuracy and speed
- Use **LSTM** or **Transformer** if you have enough data and computational resources
- Combine predictions from multiple models using ensemble methods for best results
- Always validate on out-of-sample test data with proper time series split
- Monitor for concept drift in financial models

## Additional Notes:

- **LSTM Model Training**: If using the LSTM model, ensure PyTorch is installed. The LSTM model captures long-term dependencies in the price sequence, potentially improving prediction performance.
- **Hyperparameter Tuning**: Consider tuning model hyperparameters (e.g., learning rate, batch size, number of layers) for optimal performance.
- **Ensemble Methods**: Combining predictions from multiple models (e.g., averaging, stacking) can lead to more robust and accurate predictions.
- **Feature Selection**: Regularly review and select important features to reduce model complexity and improve interpretability.
- **Model Retraining**: Periodically retrain models with new data to maintain prediction accuracy over time.

### LSTM Model

In [30]:
try:
    from src.strategy.predictive import LSTMModel

    print("Training LSTM model...")
    lstm_model = LSTMModel(epochs=30, batch_size=32, lstm_units=50)

    lstm_model.train(
        X_train, y_train,
        seq_length=60,  # Use 60-minute lookback
        X_val=X_val, y_val=y_val,
        verbose=1
    )

    # Evaluate LSTM
    trainer.models['LSTM'] = lstm_model
    trainer.evaluate_model(lstm_model, X_test, y_test)

except ImportError as e:
    print(f"PyTorch not installed: {e}")
    print("Install with: pip install torch")

Training LSTM model...
Epoch 10/30, Loss: 0.6918
Epoch 20/30, Loss: 0.6889
Epoch 30/30, Loss: 0.6860
LSTM Model trained
Evaluating LSTM model...

LSTM - Evaluation Results
Accuracy:  0.5127
Precision: 0.5060
Recall:    0.3604
F1-Score:  0.4209

Confusion Matrix:
[[14014  7222]
 [13128  7396]]



### Transformer Model

In [31]:
try:
    from src.strategy.predictive import TransformerModel

    print("Training Transformer model...")
    transformer_model = TransformerModel(epochs=30, batch_size=32, d_model=64, num_heads=4)

    transformer_model.train(
        X_train, y_train,
        seq_length=60,  # Use 60-minute lookback
        X_val=X_val, y_val=y_val,
        verbose=1
    )

    # Evaluate Transformer
    trainer.models['Transformer'] = transformer_model
    trainer.evaluate_model(transformer_model, X_test, y_test)

except ImportError as e:
    print(f"PyTorch not installed: {e}")
    print("Install with: pip install torch")


Training Transformer model...
Epoch 10/30, Loss: 0.6969
Epoch 20/30, Loss: 0.6948
Epoch 30/30, Loss: 0.6936
Transformer Model trained
Evaluating Transformer model...

Transformer - Evaluation Results
Accuracy:  0.5092
Precision: 0.5061
Recall:    0.0568
F1-Score:  0.1022

Confusion Matrix:
[[20098  1138]
 [19358  1166]]

